# 第9章: 回帰分析による連続値予測

この Notebook は、原本 `machine-learning-book/ch09/ch09.ipynb` を最新の Python 環境と
`pytest --nbmake` による CI 実行向けに移行したものです。

原本の教育意図を保ちながら、次の点を更新しています。

- ネットワーク経由ではなく、読み取り専用サブモジュール内の `AmesHousing.txt` を直接参照する
- `mlxtend` 依存の可視化を pandas / matplotlib ベースに置き換える
- 重い処理は避けつつ、線形回帰、RANSAC、正則化回帰、多項式回帰、決定木、RandomForest を継続検証できる形にする


## この Notebook で確認すること

- 原本図版と Ames Housing データを `src/` 配下から安定して参照できることを確認する
- 単回帰と重回帰の基礎をデータ可視化とともに確認する
- 勾配降下法による線形回帰、scikit-learn の線形回帰、RANSAC を比較する
- MSE / MAE / $R^2$ による評価と、Lasso / Ridge / ElasticNet の挙動を確認する
- 多項式回帰、決定木回帰、RandomForest 回帰で非線形関係を捉えられることを確認する


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import sys

from IPython.display import Image, display
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge, RANSACRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeRegressor


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "machine-learning-book").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("リポジトリルートを見つけられませんでした。")


REPO_ROOT = find_repo_root()
CHAPTER_DIR = REPO_ROOT / "machine-learning-book" / "ch09"
DATA_PATH = CHAPTER_DIR / "AmesHousing.txt"
FIGURE_DIR = CHAPTER_DIR / "figures"

assert DATA_PATH.exists(), f"データファイルが見つかりません: {DATA_PATH}"
assert FIGURE_DIR.exists(), f"図版ディレクトリが見つかりません: {FIGURE_DIR}"

print(f"Python 実行ファイル: {sys.executable}")
print(f"Python バージョン: {platform.python_version()}")
print(f"Matplotlib バックエンド: {matplotlib.get_backend()}")
print(f"Chapter directory: {CHAPTER_DIR}")


In [ ]:
package_versions = pd.DataFrame(
    [
        ("numpy", version("numpy")),
        ("pandas", version("pandas")),
        ("matplotlib", version("matplotlib")),
        ("scikit-learn", version("scikit-learn")),
        ("pytest", version("pytest")),
    ],
    columns=["パッケージ", "バージョン"],
)
package_versions


## 原本図版の参照

原本では単回帰と重回帰の模式図が最初に掲載されています。移行版でも読み取り専用サブモジュールから
代表図版を直接参照し、`src/ch09/` 配下から壊れないことを確認します。


In [ ]:
for figure_name in ["09_01.png", "09_01_2.png"]:
    print(figure_name)
    display(Image(filename=str(FIGURE_DIR / figure_name), width=560))


## Ames Housing データの読み込み

原本は外部 URL からデータを取得していましたが、CI ではネットワーク依存を避けます。
ここではローカル同梱の `AmesHousing.txt` を読み込み、章で使う主要列だけを抽出します。


In [ ]:
columns = ["Overall Qual", "Overall Cond", "Gr Liv Area", "Central Air", "Total Bsmt SF", "SalePrice"]

df = pd.read_csv(DATA_PATH, sep="\t", usecols=columns)
df["Central Air"] = df["Central Air"].map({"N": 0, "Y": 1})
df = df.dropna(axis=0).reset_index(drop=True)

summary = pd.Series(
    {
        "samples": len(df),
        "features": len(columns) - 1,
        "missing_values_after_dropna": int(df.isnull().sum().sum()),
        "saleprice_mean": round(df["SalePrice"].mean(), 2),
        "saleprice_median": round(df["SalePrice"].median(), 2),
    }
)

display(summary.to_frame(name="値"))
display(df.head(5))


## データの可視化

原本では `mlxtend` の散布図行列とヒートマップを使っていました。移行版では pandas の
`scatter_matrix` と pandas / matplotlib ベースの相関ヒートマップに置き換えます。


In [ ]:
sample_for_plot = df.sample(n=min(len(df), 600), random_state=1)

axes = scatter_matrix(
    sample_for_plot,
    figsize=(10, 10),
    diagonal="hist",
    alpha=0.4,
    color="#4C72B0",
)
for ax in np.array(axes).ravel():
    ax.tick_params(axis="x", labelrotation=45)
plt.tight_layout()
plt.show()
plt.close("all")

corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(6.0, 4.8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()
plt.close(fig)

corr["SalePrice"].sort_values(ascending=False).to_frame(name="corr_with_saleprice")


## 勾配降下法による単回帰

原本と同様に `Gr Liv Area` から `SalePrice` を予測する単回帰を、手書きの勾配降下法クラスで学習します。
標準化を行い、損失の収束と予測値を確認します。


In [ ]:
class LinearRegressionGD:
    def __init__(self, eta: float = 0.1, n_iter: int = 80, random_state: int = 1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    def fit(self, X: np.ndarray, y: np.ndarray):
        rgen = np.random.RandomState(self.random_state)
        self.w_ = rgen.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b_ = np.array([0.0])
        self.losses_ = []

        for _ in range(self.n_iter):
            output = self.net_input(X)
            errors = y - output
            self.w_ += self.eta * 2.0 * X.T.dot(errors) / X.shape[0]
            self.b_ += self.eta * 2.0 * errors.mean()
            self.losses_.append((errors**2).mean())
        return self

    def net_input(self, X: np.ndarray) -> np.ndarray:
        return np.dot(X, self.w_) + self.b_

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.net_input(X)


X = df[["Gr Liv Area"]].to_numpy()
y = df["SalePrice"].to_numpy()

sc_x = StandardScaler()
sc_y = StandardScaler()
X_std = sc_x.fit_transform(X)
y_std = sc_y.fit_transform(y.reshape(-1, 1)).flatten()

lr_gd = LinearRegressionGD()
lr_gd.fit(X_std, y_std)

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.plot(range(1, lr_gd.n_iter + 1), lr_gd.losses_, color="#C44E52", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title("Gradient Descent Convergence")
plt.tight_layout()
plt.show()
plt.close(fig)

pred_2500 = sc_y.inverse_transform(lr_gd.predict(sc_x.transform(np.array([[2500.0]]))).reshape(-1, 1)).item()
pd.Series(
    {
        "slope_standardized": round(float(lr_gd.w_[0]), 4),
        "intercept_standardized": round(float(lr_gd.b_[0]), 4),
        "predicted_price_for_2500_sqft": round(pred_2500, 2),
    }
).to_frame(name="値")


## scikit-learn の線形回帰と正規方程式

同じ単回帰問題を `LinearRegression` と正規方程式で解き、勾配降下法の結果と整合することを確認します。


In [ ]:
def lin_regplot(ax, X_values: np.ndarray, y_values: np.ndarray, model, xlabel: str, ylabel: str) -> None:
    ax.scatter(X_values, y_values, c="steelblue", edgecolor="white", s=36, alpha=0.7)
    sort_idx = np.argsort(X_values.flatten())
    ax.plot(X_values[sort_idx], model.predict(X_values[sort_idx]), color="black", linewidth=2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


slr = LinearRegression()
slr.fit(X, y)

Xb = np.hstack((np.ones((X.shape[0], 1)), X))
w = np.linalg.inv(Xb.T @ Xb) @ Xb.T @ y

fig, ax = plt.subplots(figsize=(5.2, 3.6))
lin_regplot(ax, X, y, slr, "Living area above ground", "Sale price")
plt.tight_layout()
plt.show()
plt.close(fig)

pd.Series(
    {
        "sklearn_slope": round(float(slr.coef_[0]), 4),
        "sklearn_intercept": round(float(slr.intercept_), 4),
        "normal_equation_slope": round(float(w[1]), 4),
        "normal_equation_intercept": round(float(w[0]), 4),
    }
).to_frame(name="値")


## RANSAC によるロバスト回帰

外れ値の影響を抑えるため、原本と同様に `RANSACRegressor` を適用します。
しきい値は明示的に与え、現行 scikit-learn でも安定して再現できるようにします。


In [ ]:
ransac = RANSACRegressor(
    estimator=LinearRegression(),
    max_trials=100,
    min_samples=0.95,
    loss="absolute_error",
    residual_threshold=65000.0,
    random_state=123,
)
randsac_X = X.copy()
randsac_y = y.copy()
ransac.fit(randsac_X, randsac_y)

inlier_mask = ransac.inlier_mask_
outlier_mask = ~inlier_mask
line_X = np.arange(randsac_X.min(), randsac_X.max() + 1, 100)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(5.4, 3.8))
ax.scatter(randsac_X[inlier_mask], randsac_y[inlier_mask], c="steelblue", edgecolor="white", s=32, label="Inliers")
ax.scatter(randsac_X[outlier_mask], randsac_y[outlier_mask], c="limegreen", edgecolor="white", s=32, marker="s", label="Outliers")
ax.plot(line_X, ransac.predict(line_X), color="black", linewidth=2)
ax.set_xlabel("Living area above ground")
ax.set_ylabel("Sale price")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()
plt.close(fig)

pd.Series(
    {
        "inliers": int(inlier_mask.sum()),
        "outliers": int(outlier_mask.sum()),
        "ransac_slope": round(float(ransac.estimator_.coef_[0]), 4),
        "ransac_intercept": round(float(ransac.estimator_.intercept_), 4),
    }
).to_frame(name="値")


## 重回帰モデルの評価

章で使用する 5 特徴量すべてを使って訓練・テスト分割を行い、残差プロットと MSE / MAE / $R^2$ を確認します。


In [ ]:
target = "SalePrice"
features = [col for col in df.columns if col != target]

X_multi = df[features].to_numpy()
y_multi = df[target].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X_multi,
    y_multi,
    test_size=0.3,
    random_state=123,
)

slr_multi = LinearRegression()
slr_multi.fit(X_train, y_train)
y_train_pred = slr_multi.predict(X_train)
y_test_pred = slr_multi.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.2), sharey=True)
axes[0].scatter(y_test_pred, y_test_pred - y_test, c="limegreen", edgecolor="white", s=28)
axes[1].scatter(y_train_pred, y_train_pred - y_train, c="steelblue", edgecolor="white", s=28)
axes[0].set_title("Test residuals")
axes[1].set_title("Train residuals")
for ax in axes:
    ax.axhline(y=0.0, color="black", linewidth=1.5)
    ax.set_xlabel("Predicted values")
axes[0].set_ylabel("Residuals")
plt.tight_layout()
plt.show()
plt.close(fig)

metrics_df = pd.DataFrame(
    [
        {
            "split": "train",
            "mse": mean_squared_error(y_train, y_train_pred),
            "mae": mean_absolute_error(y_train, y_train_pred),
            "r2": r2_score(y_train, y_train_pred),
        },
        {
            "split": "test",
            "mse": mean_squared_error(y_test, y_test_pred),
            "mae": mean_absolute_error(y_test, y_test_pred),
            "r2": r2_score(y_test, y_test_pred),
        },
    ]
)
metrics_df.round(3)


## 正則化回帰

Lasso / Ridge / ElasticNet を同じ学習データに適用し、係数の疎性や汎化性能の違いを比較します。


In [ ]:
regularized_models = {
    "Lasso": Lasso(alpha=1.0, max_iter=10000, random_state=1),
    "Ridge": Ridge(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000, random_state=1),
}

rows = []
coef_table = {}
for name, model in regularized_models.items():
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    rows.append(
        {
            "model": name,
            "test_mse": mean_squared_error(y_test, test_pred),
            "test_r2": r2_score(y_test, test_pred),
            "nonzero_coefficients": int(np.count_nonzero(model.coef_)),
        }
    )
    coef_table[name] = model.coef_

display(pd.DataFrame(rows).round(3))
pd.DataFrame(coef_table, index=features).round(3)


## 多項式回帰

まず原本と同じ小さな人工データで線形回帰と 2 次回帰を比較し、その後 Ames Housing 上の非線形関係を確認します。


In [ ]:
X_poly_demo = np.array([258.0, 270.0, 294.0, 320.0, 342.0, 368.0, 396.0, 446.0, 480.0, 586.0]).reshape(-1, 1)
y_poly_demo = np.array([236.4, 234.4, 252.8, 298.6, 314.2, 342.2, 360.8, 368.0, 391.2, 390.8])

linear_demo = LinearRegression()
quadratic = PolynomialFeatures(degree=2)
quadratic_demo = LinearRegression()
X_poly_demo_quad = quadratic.fit_transform(X_poly_demo)

linear_demo.fit(X_poly_demo, y_poly_demo)
quadratic_demo.fit(X_poly_demo_quad, y_poly_demo)

X_fit_demo = np.arange(250, 600, 10).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.scatter(X_poly_demo, y_poly_demo, label="Training points", color="steelblue")
ax.plot(X_fit_demo, linear_demo.predict(X_fit_demo), linestyle="--", linewidth=2, label="Linear fit")
ax.plot(X_fit_demo, quadratic_demo.predict(quadratic.transform(X_fit_demo)), linewidth=2, label="Quadratic fit")
ax.set_xlabel("Explanatory variable")
ax.set_ylabel("Target")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()
plt.close(fig)

pd.Series(
    {
        "linear_mse": round(mean_squared_error(y_poly_demo, linear_demo.predict(X_poly_demo)), 3),
        "quadratic_mse": round(mean_squared_error(y_poly_demo, quadratic_demo.predict(X_poly_demo_quad)), 3),
        "linear_r2": round(r2_score(y_poly_demo, linear_demo.predict(X_poly_demo)), 3),
        "quadratic_r2": round(r2_score(y_poly_demo, quadratic_demo.predict(X_poly_demo_quad)), 3),
    }
).to_frame(name="値")


## Ames Housing 上の非線形関係

`Gr Liv Area` と `Overall Qual` について、線形・2 次・3 次の回帰曲線を比較します。
極端な外れ値の影響を減らすため、`Gr Liv Area < 4000` に制限します。


In [ ]:
def fit_curve_models(X_values: np.ndarray, y_values: np.ndarray):
    quadratic = PolynomialFeatures(degree=2)
    cubic = PolynomialFeatures(degree=3)
    regr = LinearRegression()
    X_quad = quadratic.fit_transform(X_values)
    X_cubic = cubic.fit_transform(X_values)

    regr.fit(X_values, y_values)
    y_lin = regr.predict(X_values)
    linear_r2 = r2_score(y_values, y_lin)

    regr.fit(X_quad, y_values)
    y_quad = regr.predict(X_quad)
    quadratic_r2 = r2_score(y_values, y_quad)

    regr.fit(X_cubic, y_values)
    y_cubic = regr.predict(X_cubic)
    cubic_r2 = r2_score(y_values, y_cubic)

    return linear_r2, quadratic_r2, cubic_r2


mask = df["Gr Liv Area"] < 4000
X_area = df.loc[mask, ["Gr Liv Area"]].to_numpy()
y_area = df.loc[mask, "SalePrice"].to_numpy()
X_qual = df[["Overall Qual"]].to_numpy()
y_qual = df["SalePrice"].to_numpy()

area_scores = fit_curve_models(X_area, y_area)
qual_scores = fit_curve_models(X_qual, y_qual)

pd.DataFrame(
    [
        {"feature": "Gr Liv Area", "linear_r2": area_scores[0], "quadratic_r2": area_scores[1], "cubic_r2": area_scores[2]},
        {"feature": "Overall Qual", "linear_r2": qual_scores[0], "quadratic_r2": qual_scores[1], "cubic_r2": qual_scores[2]},
    ]
).round(3)


## 決定木回帰と RandomForest 回帰

最後に、非線形性をより柔軟に扱える木ベース回帰を確認します。`nbmake` 実行時間を考慮して、
RandomForest の木の本数は原本より減らし、`n_jobs=1` にしています。


In [ ]:
tree = DecisionTreeRegressor(max_depth=3, random_state=1)
tree.fit(X_area, y_area)

sort_idx = np.argsort(X_area.flatten())
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.scatter(X_area, y_area, color="lightgray", s=24, label="Training points")
ax.plot(X_area[sort_idx], tree.predict(X_area[sort_idx]), color="black", linewidth=2, label="Tree prediction")
ax.set_xlabel("Living area above ground")
ax.set_ylabel("Sale price")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()
plt.close(fig)

forest = RandomForestRegressor(
    n_estimators=200,
    criterion="squared_error",
    random_state=1,
    n_jobs=1,
)
forest.fit(X_train, y_train)
forest_train_pred = forest.predict(X_train)
forest_test_pred = forest.predict(X_test)

pd.DataFrame(
    [
        {
            "model": "DecisionTreeRegressor",
            "train_r2": r2_score(y_area, tree.predict(X_area)),
            "test_r2": np.nan,
            "test_mae": np.nan,
        },
        {
            "model": "RandomForestRegressor",
            "train_r2": r2_score(y_train, forest_train_pred),
            "test_r2": r2_score(y_test, forest_test_pred),
            "test_mae": mean_absolute_error(y_test, forest_test_pred),
        },
    ]
).round(3)


## まとめ

この移行版 Notebook では、第9章の主要テーマである単回帰、重回帰、RANSAC、正則化回帰、
多項式回帰、決定木回帰、RandomForest 回帰を、ローカル同梱データと最新の scikit-learn API で再構成しました。

原本 `machine-learning-book/` 配下は変更せず、`src/ch09/ch09.ipynb` から読み取り専用データと図版を参照する構成にしているため、
CI 上でも継続的に回帰分析の主要コードパスを検証できます。
